# 🚀 Multi-Label Classification: Ablation Study (Kaggle T4)
EfficientNet-B0 + CBAM + ASL — Full pipeline A/B/C/D/E/F

## ⚠️ KEY FIXES (so với master gốc):
1. **evaluate.py**: KHÔNG dùng `autocast` khi evaluate → fp32 sigmoid → mAP chính xác (~+19% vs buggy)
2. **exp_C config**: Bỏ `eps: 1e-5` (gây mất gradient signal) → dùng default `eps=1e-8`
3. **exp_E, exp_F configs**: Đảm bảo không có `eps` thừa, đúng backbone/CBAM setting
4. Clone từ branch `main` đã chứa tất cả fixes


## Cell 1: Install Dependencies

In [ ]:
!pip install timm pycocotools torchmetrics scikit-learn matplotlib seaborn pyyaml tqdm -q
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")


## Cell 2: Clone Repository từ GitHub

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Thinh59/ECAAL.git'  # ← đổi nếu URL khác
REPO_DIR = Path('/kaggle/working/ECAAL')

if REPO_DIR.exists():
    print(f'Repo đã tồn tại — pulling latest...')
    os.system(f'git -C {REPO_DIR} pull')
else:
    print(f'Cloning {REPO_URL} ...')
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if ret != 0:
        raise RuntimeError('git clone thất bại! Kiểm tra REPO_URL.')

# Verify tất cả files cần thiết
required = [
    'src/train.py', 'src/losses.py', 'src/models.py',
    'src/dataset.py', 'src/evaluate.py', 'src/cbam.py', 'src/utils.py',
    'configs/exp_A_resnet_bce.yaml', 'configs/exp_B_resnet_asl.yaml',
    'configs/exp_C_efficientnet_cbam_asl.yaml', 'configs/exp_D_resnet_focal.yaml',
    'configs/exp_E_resnet_cbam_asl.yaml', 'configs/exp_F_efficientnet_asl.yaml',
]
all_ok = True
for rel in required:
    p = REPO_DIR / rel
    ok = p.exists()
    print(f"  {'OK' if ok else 'MISSING'} {rel}")
    if not ok:
        all_ok = False

if not all_ok:
    raise FileNotFoundError('Một số file thiếu — clone chưa đầy đủ!')
print('\nRepository OK!')


## Cell 3: Verify Fixes

In [ ]:
import sys, inspect
sys.path.insert(0, str(REPO_DIR / 'src'))

# ── FIX 1: Verify ASL implementation ──────────────────────────────────────
from losses import AsymmetricLoss
src = inspect.getsource(AsymmetricLoss.forward)
fix_ok = 'xs_pos_shifted = (xs_pos - self.clip).clamp(min=0)' in src
bug_ok = 'xs_neg_shifted = (xs_neg + self.clip)' not in src
print(f"  {'✅' if fix_ok else '❌'} ASL: correct probability shifting")
print(f"  {'✅' if bug_ok else '❌'} ASL: old bug absent")

# ── FIX 2: Verify evaluate.py không có autocast ───────────────────────────
from evaluate import evaluate_model
eval_src = inspect.getsource(evaluate_model)
no_autocast = 'autocast' not in eval_src
print(f"  {'✅' if no_autocast else '❌'} evaluate_model: NO autocast (fp32 evaluation)")

# ── FIX 3: Verify exp_C config không có eps=1e-5 ──────────────────────────
import yaml
cfg_c = yaml.safe_load(open(REPO_DIR / 'configs/exp_C_efficientnet_cbam_asl.yaml'))
has_eps = 'eps' in cfg_c.get('loss', {})
print(f"  {'✅' if not has_eps else '⚠️  WARNING'} exp_C config: {'no eps (correct)' if not has_eps else f'eps={cfg_c["loss"]["eps"]} present — should be removed!'}")

# ── Test ASL không NaN ────────────────────────────────────────────────────
import torch
asl = AsymmetricLoss(gamma_pos=0, gamma_neg=4, clip=0.05)
logits  = torch.tensor([[10., -10., 5., -5.]] * 8)
targets = torch.randint(0, 2, (8, 4)).float()
loss = asl(logits, targets)
print(f"  {'✅' if not torch.isnan(loss) else '❌'} ASL: no NaN on saturated logits (loss={loss.item():.4f})")

if fix_ok and bug_ok and no_autocast:
    print("\n🎉 ALL FIXES VERIFIED — Ready to train!")
else:
    raise RuntimeError("❌ Một số fix chưa được apply — kiểm tra lại repo!")


## Cell 4: Setup Directories

In [ ]:
os.makedirs('/kaggle/working/data/coco_subset', exist_ok=True)
os.makedirs('/kaggle/working/outputs', exist_ok=True)
os.chdir('/kaggle/working')
print(f"Working dir: {os.getcwd()}")
print("✅ Directory structure created")


## Cell 5: Dataset Preparation

**Trước khi chạy:** Notebook → **+ Add Data** → search `coco-2017-dataset` (awsaf49) → Add.
Dataset sẽ mount tại `/kaggle/input/coco-2017-dataset/`.


In [ ]:
# Verify COCO dataset
coco_root = '/kaggle/input/coco-2017-dataset'
if os.path.exists(coco_root):
    print("✅ COCO 2017 dataset found")
    for name, path in [
        ('train annotations', f'{coco_root}/annotations/instances_train2017.json'),
        ('val annotations',   f'{coco_root}/annotations/instances_val2017.json'),
        ('train2017/',        f'{coco_root}/train2017'),
        ('val2017/',          f'{coco_root}/val2017'),
    ]:
        print(f"  {'OK' if os.path.exists(path) else 'MISSING'} {name}")
else:
    print(f"❌ COCO dataset not found at {coco_root}")
    print("   → Notebook → + Add Data → coco-2017-dataset (awsaf49)")


## Cell 6: Create COCO 2017 Subset (16k train + 4k val)

In [ ]:
import json, random
import numpy as np
from collections import defaultdict
from pathlib import Path

SUBSET_DIR = Path('/kaggle/working/data/coco_subset')
train_f = SUBSET_DIR / 'subset_train_ids.json'
val_f   = SUBSET_DIR / 'subset_val_ids.json'

if train_f.exists() and val_f.exists():
    print(f"✅ Subset đã tồn tại — skip tạo lại")
    print(f"   train: {len(json.load(open(train_f)))} ids")
    print(f"   val:   {len(json.load(open(val_f)))} ids")
else:
    sys.path.insert(0, str(REPO_DIR / 'src'))
    from dataset import create_coco_subset
    create_coco_subset(
        coco_root=coco_root,
        output_dir=str(SUBSET_DIR),
        num_train=16000,
        num_val=4000,
        seed=42,
    )
    print("✅ Subset created!")


## Cell 7: Train Experiment A — ResNet50 + BCE (Baseline)

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_A_resnet_bce.yaml


## Cell 8: Train Experiment B — ResNet50 + ASL

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_B_resnet_asl.yaml


## Cell 9: Train Experiment D — ResNet50 + Focal Loss

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_D_resnet_focal.yaml


## Cell 10: Train Experiment C — EfficientNet-B0 + CBAM + ASL ⭐ MAIN

**Expected mAP: ~0.75** (vs 0.56 trong master bị bug)

Fix so với master:
- `evaluate.py`: không dùng autocast → fp32 precision → mAP chính xác
- `exp_C config`: không có `eps=1e-5` → gradient signal đầy đủ


In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_C_efficientnet_cbam_asl.yaml


## Cell 11: Train Experiment E — ResNet50 + CBAM + ASL

Ablation: so sánh ResNet50 không CBAM (Exp B) vs có CBAM (Exp E)
Expected mAP: ~0.71-0.73


In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_E_resnet_cbam_asl.yaml


## Cell 12: Train Experiment F — EfficientNet-B0 + ASL (không CBAM)

Ablation: so sánh EffNet không CBAM (Exp F) vs có CBAM (Exp C)
Expected mAP: ~0.70-0.72


In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_F_efficientnet_asl.yaml


## Cell 13: Evaluate Results & Plot Curves

In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt

base = '/kaggle/working/outputs'
exps = {
    'A: ResNet50+BCE':       'exp_A_resnet_bce/log.json',
    'D: ResNet50+Focal':     'exp_D_resnet_focal/log.json',
    'B: ResNet50+ASL':       'exp_B_resnet_asl/log.json',
    'E: ResNet50+CBAM+ASL':  'exp_E_resnet_cbam_asl/log.json',
    'F: EffNet+ASL':         'exp_F_efficientnet_asl/log.json',
    'C: EffNet+CBAM+ASL':    'exp_C_efficientnet_cbam_asl/log.json',
}

rows = []
logs = {}
for name, rel in exps.items():
    path = os.path.join(base, rel)
    if not os.path.exists(path):
        print(f"⚠️  {name} log not found: {path}")
        continue
    records = json.load(open(path))
    logs[name] = records
    best = max(records, key=lambda r: r.get('mAP', 0))
    rows.append({
        'Experiment': name,
        'mAP': f"{best['mAP']:.4f}",
        'Macro F1': f"{best['macro_f1']:.4f}",
        'Best Epoch': best['epoch'],
    })

print("\n" + "="*65)
print("ABLATION STUDY RESULTS")
print("="*65)
print(pd.DataFrame(rows).to_string(index=False))
print("="*65)


## Cell 14: Plot Training Curves

In [ ]:
if logs:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6', '#1abc9c', '#2ecc71']

    for (name, recs), c in zip(logs.items(), colors):
        ep = [r['epoch'] for r in recs]
        ax1.plot(ep, [r['mAP'] for r in recs], marker='o', label=name, color=c, linewidth=2)
        ax2.plot(ep, [r['macro_f1'] for r in recs], marker='s', label=name, color=c, linewidth=2)

    for ax, title, ylabel in [(ax1, 'mAP vs Epoch', 'mAP'),
                               (ax2, 'Macro F1 vs Epoch', 'Macro F1')]:
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.savefig(os.path.join(base, 'ablation_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Curves saved")
else:
    print("⚠️  No log files found. Run training cells first.")


## Cell 15: Summary Statistics

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    print("\n📊 ABLATION SUMMARY:")
    print(f"  Total Experiments: {len(rows)}")
    best_exp = max(rows, key=lambda x: float(x['mAP']))
    print(f"  Best mAP: {best_exp['Experiment']} = {best_exp['mAP']}")

    # CBAM contribution (Exp C vs Exp F)
    c_row = next((r for r in rows if r['Experiment'].startswith('C:')), None)
    f_row = next((r for r in rows if r['Experiment'].startswith('F:')), None)
    if c_row and f_row:
        delta = float(c_row['mAP']) - float(f_row['mAP'])
        print(f"\n  CBAM contribution (EffNet backbone): +{delta:.4f} mAP")
        print(f"    Exp F (no CBAM): {f_row['mAP']}  →  Exp C (CBAM): {c_row['mAP']}")

    # ASL vs BCE (Exp B vs Exp A)
    a_row = next((r for r in rows if r['Experiment'].startswith('A:')), None)
    b_row = next((r for r in rows if r['Experiment'].startswith('B:')), None)
    if a_row and b_row:
        delta = float(b_row['mAP']) - float(a_row['mAP'])
        print(f"\n  ASL contribution (ResNet50 backbone): +{delta:.4f} mAP")
        print(f"    Exp A (BCE): {a_row['mAP']}  →  Exp B (ASL): {b_row['mAP']}")

    print(f"\n  Logs saved to: {base}/*/log.json")
else:
    print("No experiments completed yet.")
